# Intro and the Data Science Workflow

Before any code, a map of the territory. This notebook answers four questions.

1. What is data science, really — and how is it different from analytics or machine learning?
2. What does the day-to-day workflow actually look like?
3. Where does *this repo* sit in that workflow?
4. What does a tiny end-to-end pass feel like in practice?

## Data science vs analytics vs ML

Three terms get used interchangeably. They are not the same job.

| Field | Core question | Output |
|---|---|---|
| **Analytics** | *What happened?* | Dashboards, reports, descriptive summaries |
| **Data science** | *What is going on in this data, and what can we do with it?* | Cleaned datasets, features, hypotheses, models — the **full loop** |
| **Machine learning** | *Given clean features, what predicts the target?* | Trained models, predictions |

The headline: **most of a real ML project is not the model.** Loading, cleaning, joining, exploring, sanity-checking — that is data science. The model is the last slice of the work, and it lives in `machine-learning/`.


## The workflow loop

Textbooks draw the workflow as a straight pipeline. In practice it is a **loop** — exploration constantly sends you back to cleaning.

```
   acquire --> clean --> explore --> model --> communicate
                  ^         |          |
                  |         v          |
                  +------ loop back ---+
```

Every arrow can also point backwards. You start exploring, find a column is unusable, go back to cleaning. You train a model, the residuals look weird, go back to exploring. The loop is the work.


## Where this repo sits

This repo covers the **pre-ML layer** — acquire, clean, explore, the statistics underneath, and the feature engineering that bridges to ML.

| Stage | Where to learn it |
|---|---|
| acquire / clean / explore / communicate | **this repo** (notebooks 02–07) |
| model | `machine-learning/` |
| acquire and process at TB scale | `apache-spark/`, `databricks-data-engineer/` |
| Python language itself | `python/` |

The seven notebooks here are a slow, careful tour of the pre-ML layer before scale and modeling enter the picture.


## The PyData stack

Five libraries do almost all the work. Every notebook in this repo imports some subset of these.

| Library | Alias | Role |
|---|---|---|
| `numpy` | `np` | N-dimensional numeric arrays — the layer everything else sits on |
| `pandas` | `pd` | Labeled tables (`DataFrame`) with SQL-like operations |
| `matplotlib` | `plt` | Low-level plotting — full control over figure and axes |
| `seaborn` | `sns` | High-level statistical plots — distributions, correlations, categorical breakdowns |
| `scipy.stats` | — | Statistical tests, distributions, and inference |

For a JVM reader: think of `pandas` as what you would get if Java's `Stream<Row>` were column-typed and had a fluent SQL-like API — closest analogue is Spark's `DataFrame`. `numpy` is the layer beneath, closer to a `double[]` where arithmetic is vectorized at the C level; no explicit `for` loop required.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)


## Our running domain — Fintech Bank

Every example in this repo uses **Fintech Bank**, a mid-size digital bank operating in India. The same domain shows up in `apache-spark/`, `databricks-data-engineer/`, and `machine-learning/`, so the mental model carries across study tracks.

Four business verticals: **Cards · Loans · Accounts · Payments**.

Core tables we will use across the seven notebooks.

| Table | What it holds | Why a data scientist cares |
|---|---|---|
| `customers` | KYC info — name, age, city, income | Demographics, segmentation, missing-data patterns |
| `bank_accounts` | Account balances and types | Skewed-distribution analysis, log-transforms |
| `loan_accounts` | Loan amount, EMI, tenure | Distributions, correlation with credit score |
| `loan_repayments` | Per-loan repayment events | Time series, groupby aggregations |
| `card_transactions` | Per-card swipes | Outlier detection, hourly patterns |
| `payments` | UPI and NEFT transfers | Hypothesis testing across channels |

All datasets in these notebooks are generated **in-cell** — small enough (20 to 100 rows) to run instantly, shaped to match the real schema. Notebook 03 will introduce *how* to read CSV, JSON, Parquet, and SQL sources; the rest of the repo stays in-cell.


## A tiny end-to-end pass

We will run one short pass through the loop — acquire, look, explore, communicate — on a 20-row slice of `customers`. The point is not the code; the point is to feel the shape of the workflow before zooming in on each step in later notebooks.


In [ ]:
customers = pd.DataFrame({
    "customer_id": range(1001, 1021),
    "name": [
        "Aarav Sharma", "Diya Patel", "Vihaan Reddy", "Ananya Iyer",
        "Arjun Nair", "Ishaan Kumar", "Kavya Menon", "Reyansh Joshi",
        "Saanvi Rao", "Aditya Gupta", "Myra Singh", "Vivaan Desai",
        "Anika Bose", "Kabir Shah", "Aanya Pillai", "Dhruv Khanna",
        "Riya Verma", "Ayaan Mehta", "Sara Naidu", "Rohan Chawla",
    ],
    "age": [27, 34, 45, 29, 52, 31, 38, 26, 41, 60,
            33, 47, 28, 55, 36, 42, 30, 39, 48, 24],
    "city": [
        "Bengaluru", "Mumbai", "Hyderabad", "Chennai", "Bengaluru",
        "Pune", "Mumbai", "Bengaluru", "Hyderabad", "Delhi",
        "Mumbai", "Pune", "Chennai", "Delhi", "Bengaluru",
        "Hyderabad", "Mumbai", "Bengaluru", "Pune", "Chennai",
    ],
    "monthly_income": [
        45000, 82000, 120000, 67000, 155000, 58000, 91000, 42000,
        110000, 175000, 73000, 138000, 51000, 162000, 86000,
        125000, 64000, 98000, 142000, 39000,
    ],
})


### Step 1 — first look

The first three commands of every data science session. Run them before you do anything else.

- `head()` — what does a row look like?
- `info()` — what are the columns and their types, and are there any nulls?
- `describe()` — what is the numeric shape (min, max, mean, quartiles)?


In [ ]:
customers.head()


In [ ]:
customers.info()


In [ ]:
customers.describe()


### Step 2 — explore

A `groupby` is the workhorse of exploration. Here: average monthly income per city, sorted high to low. This is the same idea as `SELECT city, AVG(monthly_income) FROM customers GROUP BY city ORDER BY 2 DESC`.


In [ ]:
income_by_city = (
    customers
    .groupby("city")["monthly_income"]
    .mean()
    .sort_values(ascending=False)
    .round(0)
)
income_by_city


### Step 3 — communicate

A single chart turns the table above into something a stakeholder can grasp in two seconds. `seaborn` builds on `matplotlib` and handles the styling for us.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    x=income_by_city.values,
    y=income_by_city.index,
    ax=ax,
    color="steelblue",
)
ax.set_xlabel("Average monthly income (INR)")
ax.set_ylabel("City")
ax.set_title("Fintech Bank — average customer income by city")
plt.tight_layout()
plt.show()


## Mapping what we just did to the loop

| Loop stage | Cell |
|---|---|
| **acquire** | Building the `customers` DataFrame in-cell (in real life: read CSV / SQL — notebook 03) |
| **clean** | Skipped here — the data was synthetic and already clean. Notebook 04 covers missing data, dedup, type fixes |
| **explore** | `head` / `info` / `describe` / the `groupby` by city |
| **model** | Not done here — modeling lives in `machine-learning/` |
| **communicate** | The bar chart |

Notice what is missing: **no model**. That is the point. Notebooks 02 through 07 build the entire stack underneath modeling. By the time you open `machine-learning/`, the cleaning and feature engineering will already feel routine.


## What's next

Notebook 02 zooms into **NumPy** — the numeric core. Every column of every `pandas` DataFrame is, underneath, a NumPy array. Understanding `ndarray`, dtypes, vectorization, and broadcasting is what makes the rest of the stack click.

Two small exercises before you move on, to lock in the loop:

1. Add a `credit_score` column to `customers` (use `np.random.default_rng(42).integers(300, 850, 20)`) and re-run `describe()`.
2. Build a `groupby` that shows the *maximum* income per city, and plot it the same way.

Closing the notebook and writing these from memory is worth more than re-reading this page.
